# MapReduce on Local Hadoop — Student Guide

Step-by-step notebook for **creating**, **running**, and **monitoring** MapReduce jobs on the Docker Hadoop cluster.

| | |
|---|---|
| **Scenario** | **ShopStream** e-commerce — analyze product review text |
| **Prerequisite** | Cluster running — [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) Steps 1–4 |
| **Run from** | `hadoop-local-docker/` directory |
| **Written guide** | [MAPREDUCE-STUDENT-GUIDE.md](./MAPREDUCE-STUDENT-GUIDE.md) |

**How to use:** Run cells **in order**. Each step tells you what to expect before you run the next cell.

### Monitoring URLs — open these in your browser

| UI | URL |
|----|-----|
| YARN ResourceManager | http://localhost:8088 |
| YARN Proxy | http://localhost:9099 |
| MapReduce History | http://localhost:19888 |
| HDFS NameNode | http://localhost:9870 |

---
## Step 1 — What is MapReduce?

**Goal:** ShopStream wants to know which words appear most often in customer reviews.

MapReduce splits batch processing into three phases:

```mermaid
flowchart LR
  subgraph input [HDFS Input]
    L1[Review line 1]
    L2[Review line 2]
  end
  subgraph map [Map]
    M[Mapper emits word, 1]
  end
  subgraph shuffle [Shuffle]
    S[Group by word]
  end
  subgraph reduce [Reduce]
    R[Sum counts]
  end
  subgraph output [HDFS Output]
    O[part-r-00000]
  end
  L1 --> M
  L2 --> M
  M --> S --> R --> O
```

| Phase | Your code? | Example output |
|-------|------------|----------------|
| **Map** | Yes | `(delivery, 1)`, `(excellent, 1)` |
| **Shuffle** | No — Hadoop handles it | Groups all `delivery` keys |
| **Reduce** | Yes | `delivery\t2` |

---
## Step 2 — Test the algorithm locally (no Hadoop)

First prove the word-count logic works on your laptop. Hadoop only adds **distribution** across workers.

In [ ]:
from pathlib import Path
from mapreduce.local_simulation import run_wordcount, top_n

reviews_path = Path("data/ecommerce/product_reviews.txt")
lines = reviews_path.read_text(encoding="utf-8").splitlines()

print(f"Input: {len(lines)} review lines")
print("--- First 3 lines ---")
for line in lines[:3]:
    print(line)

In [ ]:
counts = run_wordcount(lines)

print(f"Unique words: {len(counts)}")
print("\nTop 15 words:")
for word, count in top_n(counts, 15):
    print(f"  {word}\t{count}")

**Checkpoint:** You should see words like `delivery`, `headphones`, and `customer` in the top results.

Next we rewrite this logic as scripts Hadoop can run at cluster scale.

---
## Step 3 — Mapper and reducer scripts

**Hadoop Streaming** runs separate Python programs connected by stdin/stdout:

```mermaid
sequenceDiagram
  participant IN as HDFS Input
  participant MAP as Mapper
  participant SH as Shuffle
  participant RED as Reducer
  participant OUT as HDFS Output
  IN->>MAP: line via stdin
  MAP->>SH: key TAB value
  SH->>RED: sorted pairs
  RED->>OUT: final key TAB value
```

| Script | Location | Role |
|--------|----------|------|
| Mapper | `mapreduce/streaming/wordcount_mapper.py` | Emit `(word, 1)` per token |
| Reducer | `mapreduce/streaming/wordcount_reducer.py` | Sum counts per word |

> The Docker Hadoop image uses **Python 2.7**. Scripts are compatible with both Python 2.7 (cluster) and Python 3 (your laptop).

In [ ]:
print("=== MAPPER ===")
print(Path("mapreduce/streaming/wordcount_mapper.py").read_text())
print("=== REDUCER ===")
print(Path("mapreduce/streaming/wordcount_reducer.py").read_text())

### Step 3b — Test the Unix pipeline (mini-Hadoop)

This is exactly what Hadoop Streaming does. The `sort` command plays the **shuffle** step:

```
input file  →  mapper  →  sort  →  reducer  →  results
```

In [ ]:
import subprocess

sample = reviews_path.read_text(encoding="utf-8")

mapped = subprocess.run(
    ["python3", "mapreduce/streaming/wordcount_mapper.py"],
    input=sample, capture_output=True, text=True, check=True,
)
shuffled = subprocess.run(
    ["sort"], input=mapped.stdout, capture_output=True, text=True, check=True,
)
reduced = subprocess.run(
    ["python3", "mapreduce/streaming/wordcount_reducer.py"],
    input=shuffled.stdout, capture_output=True, text=True, check=True,
)

print("Top 10 from local pipeline:")
rows = sorted(
    reduced.stdout.strip().splitlines(),
    key=lambda r: int(r.split("\t")[1]), reverse=True,
)
for line in rows[:10]:
    print(line)

**Checkpoint:** Results should match Step 2. If not, fix the scripts before continuing.

---
## Step 4 — Verify the Hadoop cluster

All 7 containers must be **healthy** and 3 YARN nodes **RUNNING** before you submit jobs.

In [ ]:
!docker compose ps

In [ ]:
!docker exec resourcemanager yarn node -list

**Checkpoint:** 7 healthy containers, 3 YARN nodes in `RUNNING` state.

If not, go back to [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) and restart the cluster.

---
## Step 5 — Upload input data to HDFS

MapReduce reads from **HDFS**, not your laptop filesystem.

In [ ]:
!docker exec namenode hdfs dfs -mkdir -p /shopstream/raw/reviews
!docker exec namenode hdfs dfs -mkdir -p /shopstream/processed

!docker cp data/ecommerce/product_reviews.txt namenode:/tmp/product_reviews.txt
!docker exec namenode hdfs dfs -put -f /tmp/product_reviews.txt /shopstream/raw/reviews/product_reviews.txt

!docker exec namenode hdfs dfs -ls /shopstream/raw/reviews
!echo "--- Preview ---"
!docker exec namenode hdfs dfs -cat /shopstream/raw/reviews/product_reviews.txt | head -5

**Checkpoint:** File appears at `/shopstream/raw/reviews/product_reviews.txt`.

---
## Step 6 — Deploy and submit your MapReduce job

```mermaid
flowchart TB
  A[Laptop: mapper.py + reducer.py] -->|docker cp| B[namenode container]
  C[HDFS input file] --> D[hadoop-streaming JAR]
  B --> D
  D -->|YARN schedules| E[Map tasks on workers]
  E --> F[Shuffle]
  F --> G[Reduce tasks]
  G --> H[HDFS output part-r-*]
  D --> I[Monitor: localhost:8088]
  G --> J[History: localhost:19888]
```

1. Deploy scripts into the cluster
2. Delete any previous output directory
3. Submit via `hadoop-streaming` JAR
4. Watch progress at http://localhost:8088/cluster/apps

In [ ]:
from mapreduce.job_helpers import deploy_scripts, submit_streaming_job, extract_tracking_url

STREAMING_DIR = "mapreduce/streaming"
INPUT_PATH = "/shopstream/raw/reviews/product_reviews.txt"
OUTPUT_PATH = "/shopstream/processed/python_wordcount"

deploy_scripts(STREAMING_DIR, ["wordcount_mapper.py", "wordcount_reducer.py"])

print("Submitting wordcount job — watch http://localhost:8088/cluster/apps")
job_log = submit_streaming_job(
    mapper="wordcount_mapper.py",
    reducer="wordcount_reducer.py",
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
)

tracking_url = extract_tracking_url(job_log)
if tracking_url:
    print("Tracking URL:", tracking_url.replace("proxyserver", "localhost"))

**Checkpoint:** Log ends with `Job ... completed successfully` and progress lines:

```
map 0% reduce 0%  →  map 100% reduce 0%  →  map 100% reduce 100%
```

In [ ]:
from mapreduce.job_helpers import read_hdfs_output

print("Top 15 word counts from HDFS:")
read_hdfs_output(OUTPUT_PATH, top_n=15)

**Checkpoint:** Results similar to Step 3b. Your Python MapReduce job ran on the cluster.

---
## Step 7 — Monitor MapReduce jobs

```mermaid
flowchart LR
  SUB[Submit job] --> YARN[YARN UI :8088]
  YARN --> RUN[RUNNING state]
  RUN --> PROXY[Proxy :9099]
  RUN --> DONE[FINISHED state]
  DONE --> HIST[History :19888]
  DONE --> HDFS[Read HDFS output]
```

In [ ]:
from mapreduce.job_helpers import list_yarn_applications, get_latest_application_id, application_status

print("=== RUNNING ===")
list_yarn_applications("RUNNING")

print("\n=== FINISHED ===")
list_yarn_applications("FINISHED")

In [ ]:
app_id = get_latest_application_id("FINISHED")
if app_id:
    print(f"Latest finished application: {app_id}")
    application_status(app_id)
else:
    print("No finished applications found yet.")

### Web UI monitoring

| Step | URL | What to check |
|------|-----|---------------|
| 1 | http://localhost:8088/cluster/apps | Application state |
| 2 | Click app → Tracking URL | Map/reduce progress |
| 3 | http://localhost:19888/jobhistory | Counters for finished jobs |
| 4 | http://localhost:9870 | Browse `/shopstream/processed/` |

### Key counters

| Counter | Meaning |
|---------|--------|
| `Map input records` | Lines read from HDFS |
| `Map output records` | Pairs emitted by mapper |
| `Reduce input groups` | Unique keys after shuffle |
| `Launched map tasks` | Parallel map containers |

In [ ]:
!docker exec historyserver mapred job -list all 2>/dev/null | tail -5 || echo "Open http://localhost:19888/jobhistory for the full list"

---
## Step 8 — Compare Python vs built-in Java wordcount

Hadoop ships a Java wordcount example. Same MapReduce pattern — different language.

In [ ]:
!docker exec namenode hdfs dfs -rm -r -f /shopstream/processed/review_wordcount_java 2>/dev/null || true

!docker exec namenode bash -lc \
  'hadoop jar $(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar | head -1) wordcount /shopstream/raw/reviews/product_reviews.txt /shopstream/processed/review_wordcount_java'

In [ ]:
print("=== Python Streaming (top 5) ===")
!docker exec namenode bash -lc "hdfs dfs -cat /shopstream/processed/python_wordcount/part-* | sort -t\$'\\t' -k2 -nr | head -5"

print("\n=== Java examples (top 5) ===")
!docker exec namenode bash -lc "hdfs dfs -cat /shopstream/processed/review_wordcount_java/part-* | sort -t\$'\\t' -k2 -nr | head -5"

**Checkpoint:** Both outputs show similar top words — your Python job matches the built-in Java example.

---
## Step 9 — Second job: sentiment keyword counts

**Question:** How many reviews contain positive vs negative keywords?

| Script | Role |
|--------|------|
| `sentiment_mapper.py` | Emit `positive\t1` or `negative\t1` when keywords match |
| `sentiment_reducer.py` | Sum counts per category |

In [ ]:
SENTIMENT_OUTPUT = "/shopstream/processed/review_sentiment"

deploy_scripts(STREAMING_DIR, ["sentiment_mapper.py", "sentiment_reducer.py"])

submit_streaming_job(
    mapper="sentiment_mapper.py",
    reducer="sentiment_reducer.py",
    input_path=INPUT_PATH,
    output_path=SENTIMENT_OUTPUT,
)

print("Sentiment results:")
read_hdfs_output(SENTIMENT_OUTPUT, top_n=10)

**Expected:** Two lines — `positive` and `negative` counts. One review can match both.

---
## Step 10 — Job lifecycle summary

```
DEVELOP (laptop)              DEPLOY (cluster)
────────────────              ────────────────
Write mapper.py               Start Hadoop cluster
Write reducer.py              Upload input → HDFS
Test: cat | map | sort | red  docker cp scripts → namenode
Fix until correct             hadoop jar streaming.jar ...
                              Monitor 8088 / 19888
                              Read part-r-* from HDFS
```

### Common errors

| Error | Fix |
|-------|-----|
| `Output directory already exists` | Delete output path before re-run |
| `Connection refused` | Wait for cluster healthy |
| Empty reducer output | Check mapper emits `key\tvalue` |
| Python error in cluster | Use provided scripts (Python 2.7 compatible) |

---
## Step 11 — Map to AWS

| Local (this lab) | AWS equivalent |
|------------------|----------------|
| HDFS paths | Amazon S3 |
| `hadoop-streaming` JAR | Amazon EMR step |
| YARN UI (:8088) | EMR console |
| History Server (:19888) | EMR logs in S3 + CloudWatch |

---

## Reference

| Resource | Path |
|----------|------|
| Written guide | [MAPREDUCE-STUDENT-GUIDE.md](./MAPREDUCE-STUDENT-GUIDE.md) |
| Module code | [mapreduce/](./mapreduce/) |
| Cluster setup | [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) |
| Full cluster tour | [Hadoop-Local-Cluster-Guide.ipynb](./Hadoop-Local-Cluster-Guide.ipynb) |